In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt
from tensorflow.keras.regularizers import l1, l2
import warnings
warnings.filterwarnings('ignore', '.*do not.*', )

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/algoo-arenaa/train.csv")
test = pd.read_csv("/kaggle/input/competitions/algoo-arenaa/test.csv")
train.head()

In [ ]:
train.info()

In [ ]:
train.isnull().sum().sort_values(ascending = False)

In [ ]:
train.describe()

In [ ]:
train["Personality"].value_counts(normalize = True)

In [ ]:
train.groupby("Personality").mean(numeric_only=True)

In [ ]:
sns.heatmap(train.corr(numeric_only=True), annot=True, cmap="coolwarm")

In [ ]:
sns.boxplot(x="Personality", y="Time_spent_Alone", data=train)

In [ ]:
test_ids = test["id"]
train = train.drop("id", axis=1)
test = test.drop("id", axis=1)

In [ ]:
def add_features(df):
    df["Social_index"] = (
        df["Social_event_attendance"] +
        df["Going_outside"] +
        df["Friends_circle_size"] +
        df["Post_frequency"]
    ) - df["Time_spent_Alone"]

    df["social_ratio"] = (
        df["Social_event_attendance"] /
        (df["Time_spent_Alone"] + 1)
    )

    df["outside_social"] = (
        df["Going_outside"] * df["Friends_circle_size"]
    )

    df["digital_vs_real"] = (
        df["Post_frequency"] - df["Social_event_attendance"]
    )

    return df

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(
    train[["Social_event_attendance",
        "Going_outside",
        "Friends_circle_size",
        "Post_frequency",
        "Time_spent_Alone"]]
)

In [ ]:
train = add_features(train)
test = add_features(test)

In [ ]:
y = train["Personality"].map({
    "Introvert": 0,
    "Extrovert": 1
})

X = train.drop("Personality", axis=1)

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


In [ ]:
X.head()

In [ ]:
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(test)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_processed.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["AUC"]
)

early_stop = EarlyStopping(
    monitor='val_AUC',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop,lr_scheduler],
    verbose=1
)

In [ ]:
val_probs = model.predict(X_val).ravel()
val_auc = roc_auc_score(y_val, val_probs)

print("Validation AUC:", val_auc)

In [ ]:
test_probs = model.predict(X_test_processed).ravel()

test_preds = (test_probs >= 0.5).astype(int)

label_map = {0: "Introvert", 1: "Extrovert"}

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "Personality": pd.Series(test_preds).map(label_map)
})

submission.to_csv("submission.csv", index=False)

In [ ]:
sub = pd.read_csv("/kaggle/working/submission.csv")

In [ ]:
sub.head()